# Chapter 15 — RLHF with TRL## PPO-Based RLHF Training Loop[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**T4/A100 GPU required for full training. Runtime: ~2-4 hours on T4.**Complete PPO-RLHF training loop using HuggingFace TRL. Smaller models (TinyLlama) run on a free Colab T4.

In [ ]:
# Check GPU
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

try:
    from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
    from transformers import AutoTokenizer, pipeline
    print('TRL and Transformers available')
except ImportError:
    print('pip install trl transformers')

## 15.1 The Full RLHF Loop: 8 StepsFrom Chapter 15, the complete PPO-RLHF procedure:

In [ ]:
rlhf_loop = {
    'Step 1': 'Initialise policy π_θ (from SFT checkpoint)',
    'Step 2': 'Initialise reference π_ref (frozen copy of SFT)',
    'Step 3': 'Initialise reward model r_φ (from Chapter 14)',
    'Step 4': 'Sample batch of prompts x from dataset',
    'Step 5': 'Generate responses y ~ π_θ(.|x) [rollouts]',
    'Step 6': 'Score: R = r_φ(x, y) [reward model]',
    'Step 7': 'Compute KL penalties: -β·log[π_θ(y_t|c_t)/π_ref(y_t|c_t)]',
    'Step 8': 'PPO update: maximise reward - KL divergence',
}
for step, description in rlhf_loop.items():
    print(f'{step}: {description}')

## 15.2 PPOConfig: Key Hyperparameters Explained

In [ ]:
# PPOConfig walkthrough — all hyperparameters with explanations
try:
    from trl import PPOConfig
    
    config = PPOConfig(
        model_name='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
        learning_rate=1.41e-5,      # Conservative: RL training is unstable at high LR
        batch_size=16,              # Prompts per rollout batch
        mini_batch_size=4,          # Mini-batch for PPO gradient updates
        ppo_epochs=4,               # K epochs per rollout batch
        gamma=1.0,                  # No discounting: single-response episodes
        lam=0.95,                   # GAE lambda: balance bias/variance
        cliprange=0.2,              # PPO clip (ε): max policy change per step
        cliprange_value=0.2,        # Value function clip
        vf_coef=0.1,                # c_1: value function loss coefficient
        adap_kl_ctrl=True,          # Adaptive β controller
        init_kl_coef=0.2,           # Initial β (KL penalty strength)
        target=6.0,                 # Target KL divergence per step
        max_grad_norm=1.0,          # Gradient clipping
    )
    print('PPOConfig created successfully')
    print('\nKey parameters for monitoring during training:')
    print('  objective/kl        → should stay near target (6.0)')
    print('  objective/entropy   → should stay above minimum threshold')
    print('  ppo/mean_scores     → should increase over training')
    print('  ppo/val/error       → value function loss should decrease')
except Exception as e:
    print(f'PPOConfig setup: {e}')

## 15.3 KL Divergence Monitoring Dashboard

In [ ]:
import numpy as np, matplotlib.pyplot as plt

# Simulate training metrics for visualisation
np.random.seed(42)
steps = 500

# Healthy training dynamics
kl_healthy  = 6.0 + np.cumsum(np.random.normal(0, 0.2, steps)).clip(-3, 3)
reward_healthy = np.log1p(np.arange(steps)/50) * 3 + np.random.normal(0, 0.2, steps)
entropy_healthy = 4.0 - np.log1p(np.arange(steps)/200) + np.random.normal(0, 0.1, steps)

# Reward collapse scenario
reward_collapse = np.concatenate([reward_healthy[:250], reward_healthy[250:] * 1.8 + 2])
kl_collapse     = np.concatenate([kl_healthy[:250], kl_healthy[250:] + np.arange(250)*0.05])

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
titles = ['Reward Model Score', 'KL Divergence', 'Policy Entropy']
colors = ['seagreen', 'steelblue', 'purple']

for i, (metric, title, color) in enumerate(zip(
    [reward_healthy, kl_healthy, entropy_healthy], titles, colors
)):
    axes[0][i].plot(metric, color=color, linewidth=1.5)
    axes[0][i].set_title(f'Healthy: {title}')
    axes[0][i].grid(True, alpha=0.3)

axes[1][0].plot(reward_collapse, color='tomato', linewidth=1.5)
axes[1][0].axvline(250, color='black', linestyle='--', label='Collapse begins')
axes[1][0].set_title('⚠️ Reward Collapse: Score Diverges')
axes[1][0].legend()
axes[1][1].plot(kl_collapse, color='tomato', linewidth=1.5)
axes[1][1].axvline(250, color='black', linestyle='--')
axes[1][1].axhline(6.0, color='gray', linestyle=':', label='Target KL')
axes[1][1].set_title('⚠️ KL Grows Unchecked')
axes[1][1].legend()
axes[1][2].plot(entropy_healthy * 0.3 + np.random.normal(0, 0.05, steps), color='tomato', linewidth=1.5)
axes[1][2].set_title('⚠️ Mode Collapse: Entropy Drops')

for ax in axes.flat:
    ax.set_xlabel('Training Step')
plt.suptitle('RLHF Training Metrics: Healthy vs. Failure Modes', fontsize=13)
plt.tight_layout()
plt.show()

## ✅ Key Takeaways1. PPO-RLHF = rollout generation + reward scoring + KL penalty + PPO update2. Monitor KL divergence at every step: growing KL = reward hacking risk3. Adaptive KL controller (adap_kl_ctrl=True) keeps KL near target automatically4. Reward collapse: reward rises but quality falls → increase β or run human eval